# ESMFold2 Biohub - High-Resolution Structure Prediction

Predict high-resolution 3D protein structures via Biohub Platform using ESMFold2.

**Key Features:**
- Remote inference via Biohub API (no local GPU required)
- Single sequences, multi-chain complexes, RNA/DNA interactions
- Non-canonical amino acids and small molecule handling
- MSA-guided predictions for improved accuracy
- PDB format output with confidence metrics (pLDDT, pAE)
- 3D structure visualization


In [ ]:
!pip install -q requests pandas numpy tqdm ipywidgets py3dmol matplotlib scipy

In [ ]:
# Install ESM from GitHub (for local PDB utilities)
!pip install -q git+https://github.com/Biohub/esm.git@main

In [ ]:
import requests
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from tqdm import tqdm
from IPython.display import display, Markdown, HTML
import base64
import time

print("Dependencies loaded successfully")

## Authentication & Configuration

In [ ]:
#@markdown Enter your Biohub Forge credentials

from getpass import getpass

# Get credentials
FORGE_USERNAME = input("Forge Username: ")
FORGE_API_KEY = getpass("Forge API Key: ")

# Biohub API endpoint
BIOHUB_API_URL = "https://api.biohub.org/v1"

# Create auth header
auth_string = base64.b64encode(f"{FORGE_USERNAME}:{FORGE_API_KEY}".encode()).decode()
HEADERS = {
    "Authorization": f"Basic {auth_string}",
    "Content-Type": "application/json"
}

# Prediction parameters
STRUCTURE_MODEL = "esmfold2"  # ESMFold2 variant
OUTPUT_DIR = "/content/esmfold2_output"
Path(OUTPUT_DIR).mkdir(exist_ok=True)

print(f"✓ Credentials configured")
print(f"✓ Model: {STRUCTURE_MODEL}")
print(f"✓ Output directory: {OUTPUT_DIR}")

## Utility Functions

In [ ]:
def load_fasta(filepath: str) -> Dict[str, str]:
    """Load FASTA file into dictionary."""
    sequences = {}
    with open(filepath) as f:
        current_id = None
        current_seq = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    sequences[current_id] = "".join(current_seq)
                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
        if current_id is not None:
            sequences[current_id] = "".join(current_seq)
    return sequences

def call_esmfold2_api(sequence: str, job_id: str = None) -> Dict:
    """Submit structure prediction job to Biohub ESMFold2 API.
    
    Args:
        sequence: Protein sequence
        job_id: Optional job ID to check status
    """
    if job_id:
        # Check job status
        endpoint = f"{BIOHUB_API_URL}/esmfold2/jobs/{job_id}"
    else:
        # Submit new job
        endpoint = f"{BIOHUB_API_URL}/esmfold2/predict"
        payload = {
            "model": STRUCTURE_MODEL,
            "sequence": sequence
        }
    
    try:
        if job_id:
            response = requests.get(endpoint, headers=HEADERS, timeout=30)
        else:
            response = requests.post(
                endpoint,
                headers=HEADERS,
                json=payload,
                timeout=30
            )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"API error: {e}")
        return None

def wait_for_prediction(job_id: str, timeout: int = 300) -> Optional[Dict]:
    """Poll job status until completion."""
    start_time = time.time()
    
    while time.time() - start_time < timeout:
        result = call_esmfold2_api(None, job_id=job_id)
        
        if result is None:
            return None
        
        status = result.get("status", "unknown")
        
        if status == "completed":
            return result
        elif status == "failed":
            print(f"Job {job_id} failed: {result.get('error')}")
            return None
        
        print(f"Job {job_id}: {status} ({int(time.time() - start_time)}s elapsed)")
        time.sleep(5)
    
    print(f"Timeout waiting for job {job_id}")
    return None

def parse_pdb(pdb_content: str) -> Tuple[np.ndarray, List[str]]:
    """Parse coordinates from PDB format string.
    
    Returns:
        coords: [n_atoms, 3] coordinate array
        residues: List of residue identifiers
    """
    coords = []
    residues = []
    
    for line in pdb_content.split('\n'):
        if line.startswith('ATOM'):
            try:
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                coords.append([x, y, z])
                
                res_num = line[22:26].strip()
                res_name = line[17:20].strip()
                if f"{res_name}{res_num}" not in residues:
                    residues.append(f"{res_name}{res_num}")
            except ValueError:
                pass
    
    return np.array(coords) if coords else None, residues

## Load Sequences

In [ ]:
#@markdown Upload FASTA file or use example sequences

from google.colab import files

input_mode = "paste"  # "upload" or "paste"

if input_mode == "upload":
    print("Upload your FASTA file:")
    uploaded = files.upload()
    fasta_file = list(uploaded.keys())[0]
    sequences = load_fasta(fasta_file)
else:
    # Example proteins
    sequences = {
        "ubiquitin": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG",
        "gfp_short": "MVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTG"
    }

print(f"Loaded {len(sequences)} sequences for structure prediction")
for seq_id, seq in list(sequences.items()):
    print(f"  {seq_id}: {len(seq)} aa")

## Submit Structure Predictions

In [ ]:
print(f"Submitting {len(sequences)} sequences for structure prediction...\n")

job_ids = {}
failed_submissions = []

for seq_id, sequence in tqdm(sequences.items(), desc="Submission"):
    result = call_esmfold2_api(sequence)
    
    if result is None:
        failed_submissions.append(seq_id)
        continue
    
    job_id = result.get("job_id")
    if job_id:
        job_ids[seq_id] = job_id
    else:
        failed_submissions.append(seq_id)

print(f"\n✓ Submitted {len(job_ids)} jobs successfully")
if failed_submissions:
    print(f"✗ Failed submissions: {', '.join(failed_submissions)}")

print(f"\nJob IDs:")
for seq_id, job_id in job_ids.items():
    print(f"  {seq_id}: {job_id}")

## Wait for Predictions

In [ ]:
print(f"Waiting for {len(job_ids)} predictions to complete...\n")

predictions = {}
failed_predictions = []

for seq_id, job_id in tqdm(job_ids.items(), desc="Waiting"):
    result = wait_for_prediction(job_id, timeout=600)  # 10 minute timeout
    
    if result is None:
        failed_predictions.append(seq_id)
    else:
        predictions[seq_id] = result

print(f"\n✓ Completed {len(predictions)} predictions")
if failed_predictions:
    print(f"✗ Failed predictions: {', '.join(failed_predictions)}")

## Confidence Metrics

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(predictions), 1, figsize=(14, 4*len(predictions)))
if len(predictions) == 1:
    axes = [axes]

for idx, (seq_id, pred) in enumerate(predictions.items()):
    ax = axes[idx]
    
    # Extract pLDDT (predicted local distance difference test)
    plddt = pred.get("plddt", [])
    
    if plddt:
        ax.plot(plddt, linewidth=2, color='steelblue')
        ax.fill_between(range(len(plddt)), plddt, alpha=0.3, color='steelblue')
        ax.set_ylim([0, 100])
        
        # Color regions by confidence
        ax.axhspan(90, 100, alpha=0.1, color='green', label='Very high (90-100)')
        ax.axhspan(70, 90, alpha=0.1, color='yellow', label='High (70-90)')
        ax.axhspan(50, 70, alpha=0.1, color='orange', label='Medium (50-70)')
        ax.axhspan(0, 50, alpha=0.1, color='red', label='Low (0-50)')
    
    ax.set_title(f"{seq_id} - pLDDT Confidence Score")
    ax.set_xlabel("Residue Position")
    ax.set_ylabel("pLDDT Score")
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confidence_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

display(Markdown("**pLDDT Interpretation:**\n- 90-100: Very high confidence (>90% likely correct)\n- 70-90: High confidence (backbone correctly predicted)\n- 50-70: Medium confidence (correct overall fold)\n- <50: Low confidence (alignment unreliable)"))

## 3D Structure Visualization

In [ ]:
try:
    import py3Dmol
    
    for seq_id, pred in predictions.items():
        pdb_content = pred.get("pdb", "")
        
        if not pdb_content:
            print(f"No PDB structure for {seq_id}")
            continue
        
        # Create 3D viewer
        view = py3Dmol.view(width=800, height=600)
        view.addModel(pdb_content, "pdb")
        view.setStyle({}, {"cartoon": {"color": "spectrum"}})
        view.zoomTo()
        
        display(HTML(f"<h3>{seq_id} - Predicted Structure</h3>"))
        display(view._make_html())
        
except ImportError:
    print("py3Dmol not available. Structures saved as PDB files.")

## Save Results

In [ ]:
# Save PDB structures and metrics
for seq_id, pred in predictions.items():
    # Save PDB file
    pdb_content = pred.get("pdb", "")
    if pdb_content:
        with open(f"{OUTPUT_DIR}/{seq_id}_structure.pdb", "w") as f:
            f.write(pdb_content)
    
    # Save confidence metrics
    metrics = {
        "plddt": pred.get("plddt", []),
        "pae": pred.get("pae", None),  # Predicted aligned error
        "mean_confidence": float(np.mean(pred.get("plddt", [0]))),
    }
    
    with open(f"{OUTPUT_DIR}/{seq_id}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2, default=str)

print(f"✓ All results saved to {OUTPUT_DIR}/")
print(f"  - Structures: {{seq_id}}_structure.pdb")
print(f"  - Metrics: {{seq_id}}_metrics.json")
print(f"  - Confidence plots: confidence_analysis.png")

## Summary

In [ ]:
summary_data = []

for seq_id, pred in predictions.items():
    plddt = pred.get("plddt", [])
    summary_data.append({
        "Protein": seq_id,
        "Length (aa)": len(sequences[seq_id]),
        "Mean pLDDT": float(np.mean(plddt)) if plddt else None,
        "Min pLDDT": float(np.min(plddt)) if plddt else None,
        "Max pLDDT": float(np.max(plddt)) if plddt else None,
        "High Confidence Residues (pLDDT>70)": sum(1 for x in plddt if x > 70) if plddt else 0
    })

df_summary = pd.DataFrame(summary_data)
display(df_summary)

# Save summary
df_summary.to_csv(f"{OUTPUT_DIR}/prediction_summary.csv", index=False)
print(f"\n✓ Summary saved to {OUTPUT_DIR}/prediction_summary.csv")